### **SECTION 1 - IMPORTS**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.3 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
import pandas as pd
import jiwer

### **SECTION 2 - DATA LOADING**

In [ ]:
train_csv_path = Path("/content/drive/MyDrive/DL/train.csv")
val_csv_path   = Path("/content/drive/MyDrive/DL/val.csv")

train_df = pd.read_csv(train_csv_path)
val_df   = pd.read_csv(val_csv_path)

### **SECTION 3 - VOCABULARY BUILDER**

In [ ]:
def build_vocab(labels):
    """
    Builds character ↔ index mappings from all unique characters in the dataset.

    Args:
        labels (list[str]): Raw ground-truth label strings.

    Returns:
        vocab     (dict): char  → int  (1-indexed; 0 reserved for blank)
        inv_vocab (dict): int   → char (0 maps to empty string "")
    """
    # Collect every unique character across all labels
    chars = sorted({c for s in labels for c in str(s)})

    # Characters start at index 1 — index 0 is reserved for the CTC blank token
    vocab     = {c: i + 1 for i, c in enumerate(chars)}
    inv_vocab = {i: c for c, i in vocab.items()}
    inv_vocab[0] = ""   # Blank token produces no output character

    return vocab, inv_vocab


vocab, inv_vocab = build_vocab(train_df["label"].tolist())

# num_classes = unique characters + 1 blank token
num_classes = len(vocab) + 1

# Used later to store the longest label length in the checkpoint
max_len = int(train_df["label"].astype(str).map(len).max())

print(f"Vocabulary size : {len(vocab)} characters")
print(f"num_classes     : {num_classes}  (includes blank)")
print(f"Longest label   : {max_len} characters")

Vocabulary size : 19 characters
num_classes     : 20  (includes blank)
Longest label   : 7 characters


### **SECTION 4 - DATASET & COLLATION**

In [ ]:
class PlateDataset(Dataset):
    """
    Loads license plate images and their corresponding character-sequence labels.

    Each sample returns:
        image   (Tensor [C, H, W]): Preprocessed plate image.
        targets (IntTensor [L]):    Label encoded as integer indices.
        label   (str):              Raw ground-truth string (for evaluation).
        path    (str):              Relative image path (for result logging).
    """

    def __init__(self, data_csv_path, vocab, transform, root_dir=None):
        self.df        = pd.read_csv(data_csv_path)
        self.vocab     = vocab
        self.transform = transform
        self.root_dir  = Path(root_dir) if root_dir else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load and convert image to RGB (handles grayscale or RGBA source files)
        img_path = self.root_dir / row["image_path"]
        image    = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label   = str(row["label"])
        targets = torch.IntTensor([self.vocab[c] for c in label])

        return image, targets, label, str(row["image_path"])


def collate_fn(batch):
    """
    Packs a variable-length batch into the flat format required by nn.CTCLoss.

    CTCLoss expects:
        - log_probs    : [T, B, C]  — model output (time × batch × classes)
        - targets      : [sum(L_i)] — all label sequences concatenated into one 1-D tensor
        - input_lengths: [B]        — number of time steps per sample
        - target_lengths: [B]       — number of characters per label

    Args:
        batch (list): Each element is (image, targets, label, path).

    Returns:
        images         (Tensor [B, C, H, W])
        targets        (IntTensor [sum(L_i)])
        target_lengths (IntTensor [B])
        labels         (tuple[str])
        paths          (tuple[str])
    """
    images, targets, labels, paths = zip(*batch)

    images         = torch.stack(images)
    target_lengths = torch.IntTensor([len(t) for t in targets])
    targets        = torch.cat(targets)  # Flatten into a single 1-D sequence

    return images, targets, target_lengths, labels, paths

### **SECTION 5 - MODEL ARCHITECTURE  (ResNet50_CRNN)**

In [ ]:
class ResNet50_CRNN(nn.Module):
    """
    Convolutional Recurrent Neural Network using a ResNet50 backbone.

    Args:
        num_classes (int): Total output classes = vocabulary size + 1 (blank).
    """

    def __init__(self, num_classes):
        super().__init__()

        # --- 1. CNN Encoder ---
        # Load ImageNet-pretrained ResNet50 and strip the average pool + FC head.
        # Output: [B, 2048, H', W']  where H'≈3, W'≈12 for 96×384 input.
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])

        # --- 2. Sequence Modeler ---
        # BiLSTM reads the width-axis slices left-to-right and right-to-left,
        # capturing both prefix and suffix context for each character position.
        # Input size = 2048 (ResNet50 output channels).
        # Hidden size = 256 per direction → 512 concatenated.
        self.rnn = nn.LSTM(
            input_size=2048,
            hidden_size=256,
            bidirectional=True,
            batch_first=True,
            num_layers=2,
            dropout=0.3     # Reduced from 0.5 in Baseline 3
        )

        # --- 3. Classifier Head ---
        # Maps the 512-dim LSTM output to a score for each character class.
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # 1. Extract spatial features
        features = self.encoder(x)                                               # [B, 2048, H', W']

        # 2. Collapse the height axis to a single row - plates are horizontal
        features = torch.nn.functional.adaptive_avg_pool2d(features, (1, None))  # [B, 2048, 1, W']

        # 3. Remove the singleton height dimension
        features = features.squeeze(2)                                           # [B, 2048, W']

        # 4. Rearrange to (Batch, Time, Features) for the LSTM
        features = features.permute(0, 2, 1)                                     # [B, W', 2048]

        # 5. Bidirectional LSTM reads the character sequence
        out, _ = self.rnn(features)                                              # [B, W', 512]

        # 6. Project to character class scores
        logits = self.fc(out)                                                    # [B, W', num_classes]

        # 7. Reshape to [T, B, C] and convert to log-probabilities for CTCLoss
        return logits.permute(1, 0, 2).log_softmax(2)                            # [W', B, num_classes]

### **SECTION 6 - DECODING & EVALUATION UTILITIES**

In [ ]:
def decode_predictions(log_probs, inv_vocab):
    """
    Greedy CTC decoder: takes the most likely character at each time step,
    then removes blank tokens and consecutive duplicate characters.

    Args:
        log_probs (Tensor [T, B, C]): Model log-probabilities.
        inv_vocab (dict):             Index → character mapping.

    Returns:
        list[str]: One decoded string per sample in the batch.
    """
    pred_ids = log_probs.argmax(dim=2).permute(1, 0).cpu().tolist()  # [B, T]

    decoded_texts = []
    for ids in pred_ids:
        result, prev = [], -1
        for idx in ids:
            # Keep character only if it isn't the blank token and isn't a repeat
            if idx != prev and idx != 0:
                result.append(inv_vocab[idx])
            prev = idx
        decoded_texts.append("".join(result))

    return decoded_texts


def evaluate(model, loader, device, inv_vocab):
    """
    Computes Exact Match Accuracy and mean Character Error Rate (CER)
    over an entire DataLoader split.

    Returns:
        accuracy (float): Proportion of plates decoded with zero errors.
        mean_cer (float): Average CER across all samples (lower is better).
    """
    model.eval()
    exact, total, cumulative_cer = 0, 0, 0.0

    with torch.no_grad():
        for images, _, _, gt_labels, _ in loader:
            images      = images.to(device)
            log_probs   = model(images)
            pred_labels = decode_predictions(log_probs, inv_vocab)

            for pred, gt in zip(pred_labels, gt_labels):
                total          += 1
                exact          += int(pred == gt)
                cumulative_cer += jiwer.cer(gt, pred)

    accuracy = exact / max(total, 1)
    mean_cer = cumulative_cer / max(total, 1)
    return accuracy, mean_cer


def save_best_validation_results(model, loader, device, inv_vocab, save_path):
    """
    Runs inference over the validation set and writes a CSV of predictions.

    Output columns: image_path | label | predicted_label

    Args:
        model     : Trained CRNN model.
        loader    : Validation DataLoader.
        device    : torch.device.
        inv_vocab : Index → character mapping.
        save_path : pathlib.Path — destination CSV file.
    """
    model.eval()
    results = []

    print(f"Generating predictions → {save_path.name} ...")

    with torch.no_grad():
        for images, _, _, gt_labels, paths in loader:
            images      = images.to(device)
            log_probs   = model(images)
            pred_labels = decode_predictions(log_probs, inv_vocab)

            for path, gt, pred in zip(paths, gt_labels, pred_labels):
                results.append({
                    "image_path":      path,
                    "label":           gt,
                    "predicted_label": pred
                })

    pd.DataFrame(results).to_csv(save_path, index=False)
    print(f"Saved to: {save_path}")

### **SECTION 7 - TRAINING CONFIGURATION & DATA PIPELINE**

In [ ]:
# ── 7.1  Device ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


In [ ]:
# ── 7.2  Transforms ──────────────────────────────────────────────────────────
#
# KEY CHANGE from Baseline 3:
#   Separate pipelines for train and val.
#   Validation images are NOT augmented - applying ColorJitter/RandomAffine
#   to the val set was artificially inflating CER in Baseline 3 because the
#   model was being evaluated on distorted inputs.
#
# Input size increased from 64×320 → 96×384:
#   The extra pixels preserve finer stroke details that help distinguish
#   ambiguous character pairs (e.g., 'B' vs '8', '0' vs 'O').

train_transform = transforms.Compose([
    transforms.Resize((96, 384)),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.3, contrast=0.3),           # Lighting variation
        transforms.RandomAffine(degrees=3, translate=(0.05, 0.05)),     # Camera tilt/shift
    ], p=0.7),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
])

val_transform = transforms.Compose([
    transforms.Resize((96, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# ── 7.3  Datasets & DataLoaders ───────────────────────────────────────────────
image_root_dir = train_csv_path.parent

train_ds = PlateDataset(train_csv_path, vocab, train_transform, root_dir=image_root_dir)
val_ds   = PlateDataset(val_csv_path,   vocab, val_transform,   root_dir=image_root_dir)

# Batch size doubled from 16 → 32 for more stable gradient estimates.
# collate_fn handles the variable-length label padding required by CTCLoss.
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
# ── 7.4  Model, Loss, Optimizer, Scheduler ───────────────────────────────────
model     = ResNet50_CRNN(num_classes).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

# Halve LR if validation CER does not improve for 5 consecutive epochs.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 143MB/s]


### **SECTION 8 - TRAINING LOOP**

In [ ]:
EPOCHS     = 50
TIME_STEPS = None   # Captured from the first forward pass; used in checkpoint
best_cer   = 1.0    # Start at worst possible CER; updated whenever a new best is found

CHECKPOINT_PATH = "/content/drive/MyDrive/DL/resnet50_crnn_best.pt"

for epoch in range(1, EPOCHS + 1):

    # ── 8.1  Training Phase ────────────────────────────────────────────────
    model.train()
    running_loss = 0.0

    for images, targets, target_lengths, _, _ in train_loader:
        images  = images.to(device)
        targets = targets.to(device)

        # Forward pass → [T, B, num_classes] log-probabilities
        log_probs = model(images)

        # Capture the number of time steps from the first batch
        if TIME_STEPS is None:
            TIME_STEPS = log_probs.size(0)

        # Tell CTCLoss how many time steps were produced per image
        input_lengths = torch.full(
            (images.size(0),), log_probs.size(0), dtype=torch.long
        ).to(device)

        loss = criterion(log_probs, targets, input_lengths, target_lengths)

        optimizer.zero_grad()
        loss.backward()

        # KEY CHANGE from Baseline 3:
        # Clip gradients to prevent the NaN/Inf spikes observed with CTC loss.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()
        running_loss += loss.item()

    # ── 8.2  Validation Phase ─────────────────────────────────────────────
    val_acc, val_cer = evaluate(model, val_loader, device, inv_vocab)
    scheduler.step(val_cer)

    avg_loss = running_loss / len(train_loader)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Val CER: {val_cer:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # ── 8.3  Checkpoint - Save Best Model ─────────────────────────────────
    if val_cer < best_cer:
        best_cer = val_cer
        torch.save({
            "model_state": model.state_dict(),
            "inv_vocab":   inv_vocab,
            "vocab":       vocab,
            "num_classes": num_classes,
            "max_len":     max_len,
            "time_steps":  TIME_STEPS,
            "img_height":  96,
            "img_width":   384,
            "backbone":    "resnet50",
        }, CHECKPOINT_PATH)
        print(f"  ✓ New best CER: {best_cer:.4f} - checkpoint saved.")

print(f"\nTraining complete. Best Validation CER: {best_cer:.4f}")

Epoch 01/50 | Train Loss: 3.6062 | Val CER: 1.0000 | Val Acc: 0.0000
Epoch 02/50 | Train Loss: 2.9623 | Val CER: 1.0000 | Val Acc: 0.0000
Epoch 03/50 | Train Loss: 2.8201 | Val CER: 1.0000 | Val Acc: 0.0000
Epoch 04/50 | Train Loss: 2.6764 | Val CER: 0.9865 | Val Acc: 0.0000
  ✓ New best CER: 0.9865 - checkpoint saved.
Epoch 05/50 | Train Loss: 2.4268 | Val CER: 0.9569 | Val Acc: 0.0000
  ✓ New best CER: 0.9569 - checkpoint saved.
Epoch 06/50 | Train Loss: 2.0808 | Val CER: 0.6641 | Val Acc: 0.0000
  ✓ New best CER: 0.6641 - checkpoint saved.
Epoch 07/50 | Train Loss: 1.6437 | Val CER: 0.4003 | Val Acc: 0.0632
  ✓ New best CER: 0.4003 - checkpoint saved.
Epoch 08/50 | Train Loss: 1.1760 | Val CER: 0.3045 | Val Acc: 0.1158
  ✓ New best CER: 0.3045 - checkpoint saved.
Epoch 09/50 | Train Loss: 0.8610 | Val CER: 0.2388 | Val Acc: 0.2000
  ✓ New best CER: 0.2388 - checkpoint saved.
Epoch 10/50 | Train Loss: 0.6850 | Val CER: 0.2143 | Val Acc: 0.2526
  ✓ New best CER: 0.2143 - checkpoint sa

### **SECTION 9 - DOWNLOAD CHECKPOINT**

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **SECTION 10 - SAVE VALIDATION PREDICTIONS TO CSV**

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.to(device)

drive_save_path = Path("/content/drive/MyDrive/DL/val_baseline3_improved_results.csv")

save_best_validation_results(
    model     = model,
    loader    = val_loader,
    device    = device,
    inv_vocab = inv_vocab,
    save_path = drive_save_path,
)

Generating predictions → val_baseline3_improved_results.csv ...
Saved to: /content/drive/MyDrive/DL/val_baseline3_improved_results.csv
